In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Load data
df = pd.read_csv("../data/Sample - Superstore.csv",
                 parse_dates=["Order Date", "Ship Date"],
                 encoding="latin-1")

# Time features
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["YearMonth"] = df["Order Date"].dt.to_period("M")

# Business metrics -- profit margin per row
# This gives each transaction its own margin score
df["Profit_Margin"] = (df["Profit"] / df["Sales"] * 100).round(2)

# Anomaly detection pipeline
features = ["Sales", "Quantity", "Discount", "Profit"]
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest = IsolationForest(contamination=0.01,
                             random_state=42,
                             n_estimators=100)
iso_forest.fit(X_scaled)

df["Anomaly"] = iso_forest.predict(X_scaled)
df["Anomaly_Score"] = iso_forest.decision_function(X_scaled)

# Label anomaly types as we did in Phase 2
df["Anomaly_Type"] = "Normal"
df.loc[(df["Anomaly"] == -1) & (df["Profit"] > 0), "Anomaly_Type"] = "Opportunity"
df.loc[(df["Anomaly"] == -1) & (df["Profit"] <= 0), "Anomaly_Type"] = "Risk"

print("Setup complete. Shape:", df.shape)
print("\nAnomaly type counts:")
print(df["Anomaly_Type"].value_counts())

Setup complete. Shape: (9994, 28)

Anomaly type counts:
Anomaly_Type
Normal         9894
Opportunity      68
Risk             32
Name: count, dtype: int64


In [2]:
# Calculate regional business metrics from Phase 1
region_metrics = df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean"),
    Transaction_Count=("Sales", "count")
).round(2)

region_metrics["Profit_Margin"] = (
    region_metrics["Total_Profit"] /
    region_metrics["Total_Sales"] * 100
).round(2)

# Count risk anomalies per region from Phase 2
risk_counts = df[df["Anomaly_Type"] == "Risk"].groupby(
    "Region").size().rename("Risk_Anomalies")

opportunity_counts = df[df["Anomaly_Type"] == "Opportunity"].groupby(
    "Region").size().rename("Opportunity_Anomalies")

# Combine both signals into one regional summary table
regional_alert = region_metrics.join(risk_counts).join(opportunity_counts)

# Fill any region that had zero anomalies with 0 instead of NaN
regional_alert["Risk_Anomalies"] = regional_alert[
    "Risk_Anomalies"].fillna(0).astype(int)
regional_alert["Opportunity_Anomalies"] = regional_alert[
    "Opportunity_Anomalies"].fillna(0).astype(int)

print("=== Regional Risk Summary ===\n")
print(regional_alert[["Total_Sales", "Total_Profit", "Profit_Margin",
                       "Risk_Anomalies", "Opportunity_Anomalies"]])

=== Regional Risk Summary ===

         Total_Sales  Total_Profit  Profit_Margin  Risk_Anomalies  \
Region                                                              
Central    501239.89      39706.36           7.92              10   
East       678781.24      91522.78          13.48              12   
South      391721.90      46749.43          11.93               9   
West       725457.82     108418.45          14.94               1   

         Opportunity_Anomalies  
Region                          
Central                     11  
East                        28  
South                       12  
West                        17  


In [3]:
alerts = []

for region, row in regional_alert.iterrows():

    # Determine margin health
    if row["Profit_Margin"] < 5:
        margin_status = "critically low"
        margin_severity = 3
    elif row["Profit_Margin"] < 10:
        margin_status = "below average"
        margin_severity = 2
    else:
        margin_status = "healthy"
        margin_severity = 1

    # Determine anomaly risk level
    if row["Risk_Anomalies"] >= 10:
        anomaly_status = "high"
        anomaly_severity = 3
    elif row["Risk_Anomalies"] >= 5:
        anomaly_status = "moderate"
        anomaly_severity = 2
    else:
        anomaly_status = "low"
        anomaly_severity = 1

    # Combined severity score -- both signals weighted equally
    combined_severity = margin_severity + anomaly_severity

    # Generate plain English alert
    alert = {
        "Region": region,
        "Profit_Margin": row["Profit_Margin"],
        "Risk_Anomalies": row["Risk_Anomalies"],
        "Opportunity_Anomalies": row["Opportunity_Anomalies"],
        "Combined_Severity": combined_severity,
        "Alert": (
            f"Region {region} has a {margin_status} profit margin of "
            f"{row['Profit_Margin']}% with {anomaly_status} anomaly risk "
            f"({int(row['Risk_Anomalies'])} flagged transactions). "
            f"Opportunities detected: {int(row['Opportunity_Anomalies'])}."
        )
    }
    alerts.append(alert)

# Convert to dataframe and sort by severity -- worst first
alerts_df = pd.DataFrame(alerts).sort_values(
    "Combined_Severity", ascending=False)

print("=== SentinelIQ Regional Alerts (Ranked by Severity) ===\n")
for _, row in alerts_df.iterrows():
    print(f"[Severity {row['Combined_Severity']}] {row['Alert']}")
    print()

=== SentinelIQ Regional Alerts (Ranked by Severity) ===

[Severity 5] Region Central has a below average profit margin of 7.92% with high anomaly risk (10 flagged transactions). Opportunities detected: 11.

[Severity 4] Region East has a healthy profit margin of 13.48% with high anomaly risk (12 flagged transactions). Opportunities detected: 28.

[Severity 3] Region South has a healthy profit margin of 11.93% with moderate anomaly risk (9 flagged transactions). Opportunities detected: 12.

[Severity 2] Region West has a healthy profit margin of 14.94% with low anomaly risk (1 flagged transactions). Opportunities detected: 17.



In [4]:
# Category business metrics
category_metrics = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean"),
    Transaction_Count=("Sales", "count")
).round(2)

category_metrics["Profit_Margin"] = (
    category_metrics["Total_Profit"] /
    category_metrics["Total_Sales"] * 100
).round(2)

# Anomaly counts per category
risk_by_cat = df[df["Anomaly_Type"] == "Risk"].groupby(
    "Category").size().rename("Risk_Anomalies")

opportunity_by_cat = df[df["Anomaly_Type"] == "Opportunity"].groupby(
    "Category").size().rename("Opportunity_Anomalies")

category_alert = category_metrics.join(risk_by_cat).join(opportunity_by_cat)
category_alert["Risk_Anomalies"] = category_alert[
    "Risk_Anomalies"].fillna(0).astype(int)
category_alert["Opportunity_Anomalies"] = category_alert[
    "Opportunity_Anomalies"].fillna(0).astype(int)

# Generate category alerts
cat_alerts = []

for category, row in category_alert.iterrows():

    if row["Profit_Margin"] < 5:
        margin_severity = 3
        margin_status = "critically low"
    elif row["Profit_Margin"] < 10:
        margin_severity = 2
        margin_status = "below average"
    else:
        margin_severity = 1
        margin_status = "healthy"

    if row["Risk_Anomalies"] >= 10:
        anomaly_severity = 3
        anomaly_status = "high"
    elif row["Risk_Anomalies"] >= 5:
        anomaly_severity = 2
        anomaly_status = "moderate"
    else:
        anomaly_severity = 1
        anomaly_status = "low"

    combined_severity = margin_severity + anomaly_severity

    cat_alerts.append({
        "Category": category,
        "Profit_Margin": row["Profit_Margin"],
        "Risk_Anomalies": row["Risk_Anomalies"],
        "Combined_Severity": combined_severity,
        "Alert": (
            f"{category} has a {margin_status} profit margin of "
            f"{row['Profit_Margin']}% with {anomaly_status} anomaly risk "
            f"({int(row['Risk_Anomalies'])} flagged transactions). "
            f"Opportunities detected: {int(row['Opportunity_Anomalies'])}."
        )
    })

cat_alerts_df = pd.DataFrame(cat_alerts).sort_values(
    "Combined_Severity", ascending=False)

print("=== SentinelIQ Category Alerts (Ranked by Severity) ===\n")
for _, row in cat_alerts_df.iterrows():
    print(f"[Severity {row['Combined_Severity']}] {row['Alert']}")
    print()

=== SentinelIQ Category Alerts (Ranked by Severity) ===

[Severity 5] Furniture has a critically low profit margin of 2.49% with moderate anomaly risk (6 flagged transactions). Opportunities detected: 10.

[Severity 4] Office Supplies has a healthy profit margin of 17.04% with high anomaly risk (17 flagged transactions). Opportunities detected: 21.

[Severity 3] Technology has a healthy profit margin of 17.4% with moderate anomaly risk (9 flagged transactions). Opportunities detected: 37.



In [5]:
# Calculate overall business health metrics
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
overall_margin = (total_profit / total_sales * 100).round(2)
total_risk = len(df[df["Anomaly_Type"] == "Risk"])
total_opportunity = len(df[df["Anomaly_Type"] == "Opportunity"])

# Find most recent period in data
latest_month = df["YearMonth"].max()
latest_data = df[df["YearMonth"] == latest_month]
latest_sales = latest_data["Sales"].sum().round(2)
latest_profit = latest_data["Profit"].sum().round(2)

print("=" * 55)
print("       SENTINELIQ DAILY BUSINESS INTELLIGENCE REPORT")
print("=" * 55)

print(f"\nPERIOD ANALYSED: 2014 - 2017")
print(f"TOTAL TRANSACTIONS: {len(df)}")
print(f"OVERALL PROFIT MARGIN: {overall_margin}%")
print(f"TOTAL RISK ANOMALIES DETECTED: {total_risk}")
print(f"TOTAL OPPORTUNITY ANOMALIES DETECTED: {total_opportunity}")

print("\n--- LATEST MONTH SNAPSHOT ---")
print(f"Period: {latest_month}")
print(f"Sales:  {latest_sales}")
print(f"Profit: {latest_profit}")

print("\n--- TOP PRIORITY ALERTS ---\n")

# Show highest severity regional alert
top_region = alerts_df.iloc[0]
print(f"[REGIONAL] Severity {top_region['Combined_Severity']}")
print(f"{top_region['Alert']}")

print()

# Show highest severity category alert
top_category = cat_alerts_df.iloc[0]
print(f"[CATEGORY] Severity {top_category['Combined_Severity']}")
print(f"{top_category['Alert']}")

print()

# Show the single most anomalous transaction
worst_transaction = df[df["Anomaly_Type"] == "Risk"].nsmallest(
    1, "Anomaly_Score")
print(f"[TRANSACTION] Most anomalous risk transaction detected:")
print(worst_transaction[["Order Date", "Category", "Region",
                          "Sales", "Discount",
                          "Profit", "Anomaly_Score"]].to_string())

print("\n--- RECOMMENDATIONS ---\n")
print("1. Investigate Central region for structural margin issues.")
print("2. Review Furniture category pricing and discount strategy.")
print("3. Analyse East region Technology deals for replicable")
print("   high-profit patterns.")
print("4. Examine flagged transaction above for data integrity.")
print("\n" + "=" * 55)

       SENTINELIQ DAILY BUSINESS INTELLIGENCE REPORT

PERIOD ANALYSED: 2014 - 2017
TOTAL TRANSACTIONS: 9994
OVERALL PROFIT MARGIN: 12.47%
TOTAL RISK ANOMALIES DETECTED: 32
TOTAL OPPORTUNITY ANOMALIES DETECTED: 68

--- LATEST MONTH SNAPSHOT ---
Period: 2017-12
Sales:  83829.32
Profit: 8483.35

--- TOP PRIORITY ALERTS ---

[REGIONAL] Severity 5
Region Central has a below average profit margin of 7.92% with high anomaly risk (10 flagged transactions). Opportunities detected: 11.

[CATEGORY] Severity 5
Furniture has a critically low profit margin of 2.49% with moderate anomaly risk (6 flagged transactions). Opportunities detected: 10.

[TRANSACTION] Most anomalous risk transaction detected:
     Order Date   Category Region     Sales  Discount     Profit  Anomaly_Score
9639 2015-01-28  Furniture  South  4297.644       0.4 -1862.3124      -0.095613

--- RECOMMENDATIONS ---

1. Investigate Central region for structural margin issues.
2. Review Furniture category pricing and discount strategy